# 10 — Day-30 Early Warning — Does a Later Snapshot Help?

**Author:** Ali Čikić · General Assembly – Data Science Bootcamp

## 1. Goal

Notebooks 07–09 used a **day-14** snapshot and found only a weak signal. This notebook asks:

> Among cases whose **recorded activity span** extends beyond **day 30**, can information timestamped by day 30 help predict
> a total recorded activity span **greater than 63 days**?

Terms used in this notebook:

- **Recorded activity span** = last event timestamp minus first event timestamp. This is not necessarily the administrative
  completion time.
- **Eligible** = the recorded span extends beyond day 30. This does **not** prove that the case was formally open at day 30.
- **Target:** `long_case = 1` if `duration_days > 63` (the same exploratory threshold from `01_EDA.ipynb`, not
  recalculated).

## 2. Load the Data

In [1]:
import os
import warnings
from pathlib import Path

os.environ["PM4PY_SHOW_PROGRESS_BAR"] = "false"  # no progress bar while loading
# hide only PM4Py's optional speed-up tip ("Install the optional requirement `r4pm` ..."); other warnings still show
warnings.filterwarnings("ignore", message="Install the optional requirement `r4pm`")

import pandas as pd
import pm4py
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# the project folder: works whether Jupyter starts in the notebooks folder or in the project folder
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_root / "data" / "raw" / "BPIC15_1.xes"



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




*When PM4Py is imported it prints a short welcome and licence message. This is normal and not an error.*

In [2]:
log = pm4py.read_xes(str(data_path))
df = pm4py.convert_to_dataframe(log)

case_data = df.groupby("case:concept:name").agg(
    first_event=("time:timestamp", "min"),
    last_event=("time:timestamp", "max"),
    end_date=("case:endDate", "first"),
)
case_data["duration_days"] = (case_data["last_event"] - case_data["first_event"]).dt.total_seconds() / (24 * 60 * 60)
case_data = case_data[case_data["end_date"].notna()]   # completed cases only, as before
case_data["long_case"] = (case_data["duration_days"] > 63).astype(int)

print("Completed cases:", len(case_data))

Completed cases: 956


## 3. Rebuild the Original Train/Test Split (Notebooks 07–09)

The original split was made on the **day-14** cohort (917 cases), stratified by `long_case`, with `random_state=42`.
I rebuild it with the same code and keep it fixed. Day-30 cases are then **filtered** from these two groups:
original training cases stay in training, original test cases stay in test. Nothing is redrawn or moved.

**Why this is the same split:** `train_test_split` is given the same 917 cases in the same order (sorted by case ID),
the same `long_case` labels and the same settings as notebook 07, so it returns exactly the same two groups. The sizes and
numbers of long cases below match notebooks 07 and 08 (733 / 184 cases; 377 / 95 long).

In [3]:
day14_cases = case_data[case_data["last_event"] > case_data["first_event"] + pd.Timedelta(days=14)]

original_train_ids, original_test_ids = train_test_split(
    day14_cases.index,
    test_size=0.20,
    random_state=42,
    stratify=day14_cases["long_case"]
)

print("Original training cases:", len(original_train_ids))
print("Original test cases:", len(original_test_ids))
print("Long cases: training", day14_cases.loc[original_train_ids, "long_case"].sum(),
      "| test", day14_cases.loc[original_test_ids, "long_case"].sum())
print("Overlap:", len(set(original_train_ids) & set(original_test_ids)))

Original training cases: 733
Original test cases: 184
Long cases: training 377 | test 95
Overlap: 0


## 4. Day-30 Cohort

`prediction_time` = first event + 30 days. A case is eligible if its last event is **after** `prediction_time`.

As in notebook 07, this rule uses each case's **last** event, which is only known afterwards. The cohort is therefore
**retrospective**: *historically completed cases whose recorded activity span extends beyond day 30*. Some of these cases
already have a recorded end date on or before day 30 (counted below), so eligibility does not mean "administratively open
at day 30".

In [4]:
case_data["prediction_time"] = case_data["first_event"] + pd.Timedelta(days=30)
active = case_data[case_data["last_event"] > case_data["prediction_time"]]

# keep the original groups: training cases stay in training, test cases stay in test
train_ids = [case for case in original_train_ids if case in active.index]
test_ids = [case for case in original_test_ids if case in active.index]

**Check the cohort and the two groups.**

In [5]:
ended_by_day30 = (active["end_date"] <= active["prediction_time"]).sum()
in_original_split = set(active.index) <= set(original_train_ids) | set(original_test_ids)
train_long = active.loc[train_ids, "long_case"]
test_long = active.loc[test_ids, "long_case"]

print("Day-30 eligible cases:", len(active))
print("Eligible cases with a recorded end date on or before day 30:", ended_by_day30)
print("All of them belong to the original split:", in_original_split)
print()
print("Training cases:", len(train_ids), "| long:", train_long.sum(), "| not long:", (train_long == 0).sum())
print("Test cases:", len(test_ids), "| long:", test_long.sum(), "| not long:", (test_long == 0).sum())
print(f"Share of long cases: training {train_long.mean():.1%} | test {test_long.mean():.1%}")

Day-30 eligible cases: 838
Eligible cases with a recorded end date on or before day 30: 39
All of them belong to the original split: True

Training cases: 677 | long: 377 | not long: 300
Test cases: 161 | long: 95 | not long: 66
Share of long cases: training 55.7% | test 59.0%


## 5. Day-30 Features

Only events timestamped **on or before** each case's day-30 cutoff are used.

- `events_first30` = events by day 30
- `unique_activities_first30` = different activity codes by day 30
- `unique_resources_first30` = different resources by day 30
- `repeated_activities_first30` = `events_first30` minus `unique_activities_first30`
- `events_first15` = events with elapsed time **≤ 15 days**
- `events_days16_30` = events with elapsed time **> 15 and ≤ 30 days**
- `days_since_last_event` = days from the most recent event to the day-30 cutoff

**Two features are exact combinations of others:** `repeated_activities_first30` = `events_first30` −
`unique_activities_first30`, and `events_days16_30` = `events_first30` − `events_first15`. They add no new information.
I keep them so that the seven count features stay the same as in notebooks 07–09; their individual coefficients should not
be read as separate effects.

In [6]:
def events_by_day30(event_table, cases):
    # keep only events of eligible cases that happened on or before that case's day-30 cutoff
    events = event_table[event_table["case:concept:name"].isin(cases.index)].copy()
    events["prediction_time"] = events["case:concept:name"].map(cases["prediction_time"])
    events["first_event"] = events["case:concept:name"].map(cases["first_event"])
    first30 = events[events["time:timestamp"] <= events["prediction_time"]].copy()
    first30["day"] = (first30["time:timestamp"] - first30["first_event"]).dt.total_seconds() / (24 * 60 * 60)
    return first30

**The seven count features.** `count_columns` fixes their order; `count_features` calculates them for every case from the events kept above.

In [7]:
count_columns = [
    "events_first30",
    "unique_activities_first30",
    "unique_resources_first30",
    "repeated_activities_first30",
    "events_first15",
    "events_days16_30",
    "days_since_last_event",
]


def count_features(first30, cases):
    first30 = first30.copy()
    first30["in_first15"] = first30["day"] <= 15
    features = first30.groupby("case:concept:name").agg(
        events_first30=("concept:name", "size"),
        unique_activities_first30=("concept:name", "nunique"),
        unique_resources_first30=("org:resource", "nunique"),
        events_first15=("in_first15", "sum"),
        last_event_by_day30=("time:timestamp", "max"),
    )
    features["repeated_activities_first30"] = features["events_first30"] - features["unique_activities_first30"]
    features["events_days16_30"] = features["events_first30"] - features["events_first15"]
    features["days_since_last_event"] = (
        cases.loc[features.index, "prediction_time"] - features["last_event_by_day30"]
    ).dt.total_seconds() / (24 * 60 * 60)
    return features[count_columns]

**Apply the helpers** to the cases and check that no event after the cutoff was used.

In [8]:
first30 = events_by_day30(df, active)
counts = count_features(first30, active)
y = active.loc[counts.index, "long_case"]

print("Events recorded by day 30:", len(first30))
print("Latest event used, in days after the case start:", first30["day"].max())
print("Cases with features:", len(counts))

Events recorded by day 30: 14551
Latest event used, in days after the case start: 30.0
Cases with features: 838


### Activity indicators

Same rule as notebook 08, applied to whichever group of training cases is used: activity codes recorded by day 30 in
**10–90%** of those training cases, at most the **15 most frequent**, ties broken by activity code. The target is not used.
A 0 means "not recorded by day 30".

**Choose activity codes using training cases only.** `select_activities` returns the codes recorded in 10–90% of the given cases (at most 15, most frequent first, ties broken by code). It is only ever called with training cases.

In [9]:
def select_activities(first30, case_ids):
    # activity codes recorded in 10-90% of the given cases, at most 15, most frequent first (ties by code)
    events = first30[first30["case:concept:name"].isin(case_ids)]
    share = events.groupby("concept:name")["case:concept:name"].nunique() / len(case_ids) * 100
    share = share.rename("train_cases_pct").reset_index()
    share = share[(share["train_cases_pct"] >= 10) & (share["train_cases_pct"] <= 90)]
    share = share.sort_values(["train_cases_pct", "concept:name"], ascending=[False, True])
    return share.head(15)

**Turn the chosen activities into 0/1 columns.** `activity_flags` gives 1 if the activity was recorded by the cutoff in that case, otherwise 0.

In [10]:
def activity_flags(first30, activities, case_ids):
    # 1 if the activity was recorded by day 30 in that case, otherwise 0
    flags = pd.DataFrame(index=case_ids)
    for activity in activities:
        cases_with_activity = first30.loc[first30["concept:name"] == activity, "case:concept:name"].unique()
        flags["act_" + activity] = flags.index.isin(cases_with_activity).astype(int)
    return flags

**Build the feature table** for a list of cases: the seven counts plus the activity indicators.

In [11]:
def make_X(case_ids, activities):
    # seven counts + activity indicators for the given cases
    return counts.loc[case_ids].join(activity_flags(first30, activities, case_ids))

**Fit the approaches** with the same fixed settings every time (no tuning). Logistic Regression is a pipeline, so its scaler is fitted on the training cases only.

In [12]:
def fit_three_models(X_train, y_train):
    model_a = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
    model_b = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
    model_c = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=42).fit(X_train, y_train)
    return {"A. Majority baseline": model_a, "B. Logistic Regression": model_b, "C. Decision Tree": model_c}

**Score a fitted model.** Accuracy, precision, recall and F1 use the 0/1 predictions (threshold 0.5); ROC-AUC uses the predicted probabilities.

In [13]:
def score(model, X, y_true):
    predictions = model.predict(X)
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions),
        "F1": f1_score(y_true, predictions),
        "ROC-AUC": roc_auc_score(y_true, model.predict_proba(X)[:, 1]),
    }

## 6. Leakage Check

Change the activity and resource of every event **after** each case's day-30 cutoff (cohort and cutoffs fixed) and rebuild the
features. If they stay identical, changing post-cutoff activity and resource values does not affect the constructed
features. This does **not** prove that every possible form of leakage is absent; for example, it cannot show whether
historical timestamps reflect when the information was really available, and it does not test the retrospective cohort
selection.

In [14]:
check_activities = select_activities(first30, train_ids)["concept:name"].tolist()

changed = df.copy()
after_day30 = changed["time:timestamp"] > changed["case:concept:name"].map(active["prediction_time"])
changed.loc[after_day30, "concept:name"] = "FAKE_ACTIVITY"
changed.loc[after_day30, "org:resource"] = "FAKE_RESOURCE"

changed_first30 = events_by_day30(changed, active)
original_features = counts.join(activity_flags(first30, check_activities, counts.index))
changed_counts = count_features(changed_first30, active)
changed_flags = activity_flags(changed_first30, check_activities, counts.index)
changed_features = changed_counts.join(changed_flags)

print("Events changed:", after_day30.sum())
print("All features unchanged:", changed_features.equals(original_features))

Events changed: 26120
All features unchanged: True


## 7. Cross-Validation on the Training Cases Only

`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` on the eligible **training** cases. In every fold the activity
indicators are selected on that fold's training part only, the scaler is fitted on that part only (inside the pipeline),
and all three approaches are scored on the same validation part. The original test cases are not used here.

In other words, the model is trained and checked on **five different portions** of the training data. Activities are
selected inside each fold because selecting them with the validation cases would **contaminate** the check: the cases being
checked would help decide which features exist. That does not necessarily raise the score in every dataset, but it makes
the check less honest.

In [15]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_y = y.loc[train_ids]
train_index = pd.Index(train_ids)   # lets us turn fold positions into case IDs

cv_rows = []
validated_ids = []
for fold, (fit_index, val_index) in enumerate(folds.split(train_ids, train_y), start=1):
    fit_ids = train_index[fit_index]
    val_ids = train_index[val_index]
    validated_ids += list(val_ids)

    activities = select_activities(first30, fit_ids)["concept:name"].tolist()
    models = fit_three_models(make_X(fit_ids, activities), y.loc[fit_ids])
    for name, model in models.items():
        cv_rows.append({"Fold": fold, "Approach": name, **score(model, make_X(val_ids, activities), y.loc[val_ids])})

cv_results = pd.DataFrame(cv_rows)
print("Validated cases:", len(validated_ids), "of", len(train_ids))
print("Every training case validated:", set(validated_ids) == set(train_ids))
print("No case validated twice:", len(validated_ids) == len(set(validated_ids)))

Validated cases: 677 of 677
Every training case validated: True
No case validated twice: True


In [16]:
metric_columns = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
cv_summary = cv_results.groupby("Approach")[metric_columns].agg(["mean", "std"]).round(3)
cv_summary

Accuracy        Precision        Recall            F1  \
                           mean    std      mean    std   mean    std   mean   
Approach                                                                       
A. Majority baseline      0.557  0.002     0.557  0.002  1.000  0.000  0.715   
B. Logistic Regression    0.644  0.030     0.668  0.026  0.719  0.036  0.692   
C. Decision Tree          0.583  0.032     0.602  0.037  0.799  0.220  0.669   

                              ROC-AUC         
                          std    mean    std  
Approach                                      
A. Majority baseline    0.001   0.500  0.000  
B. Logistic Regression  0.028   0.702  0.033  
C. Decision Tree        0.093   0.629  0.047

In [17]:
cv_results.pivot(index="Fold", columns="Approach", values="ROC-AUC").round(3)

Approach,A. Majority baseline,B. Logistic Regression,C. Decision Tree
Fold,,,
1,0.5,0.755,0.613
2,0.5,0.667,0.620
3,0.5,0.707,0.666
4,0.5,0.683,0.563
5,0.5,0.698,0.683


The standard deviation shows how much the results vary **between the five folds**. It is not a confidence interval.

## 8. Final Evaluation on the Original Test Cases

The approaches, settings and features are now fixed. Activities are selected on **all** eligible training cases, each
approach is fitted once, and each is evaluated **once** on the eligible original test cases.

In [18]:
final_activities = select_activities(first30, train_ids)
train_events = first30[first30["case:concept:name"].isin(train_ids)]

# the English label recorded most often for each selected activity code
english_names = []
for activity in final_activities["concept:name"]:
    labels = train_events.loc[train_events["concept:name"] == activity, "activityNameEN"]
    english_names.append(labels.mode()[0])

activity_table = final_activities[["concept:name"]].copy()
activity_table["activityNameEN"] = english_names
activity_table["train_cases_pct"] = final_activities["train_cases_pct"]
activity_table.round(1).reset_index(drop=True)

,concept:name,activityNameEN,train_cases_pct
0,01_HOOFD_030_1,send confirmation receipt,89.1
1,01_HOOFD_030_2,enter senddate acknowledgement,87.7
2,01_HOOFD_101,registration date publication,69.4
3,01_HOOFD_011,OLO messaging active,63.5
4,02_DRZ_010,forward to the competent authority,63.1
5,04_BPT_005,regular procedure without MER,62.0
6,01_HOOFD_065_2,enter senddate procedure confirmation,49.9
7,01_HOOFD_061,start WABOprocedure,48.7
8,01_HOOFD_065_1,send procedure confirmation,44.5
9,01_HOOFD_065_0,create procedure confirmation,38.3


In [19]:
activities = final_activities["concept:name"].tolist()
X_train = make_X(train_ids, activities)
X_test = make_X(test_ids, activities)
y_train = y.loc[train_ids]
y_test = y.loc[test_ids]

final_models = fit_three_models(X_train, y_train)

test_results = {}
for name, model in final_models.items():
    test_results[name] = score(model, X_test, y_test)
    test_results[name]["Cases flagged long"] = int(model.predict(X_test).sum())
test_results = pd.DataFrame(test_results).T
test_results["Cases flagged long"] = test_results["Cases flagged long"].astype(int)

print("Test cases:", len(y_test), "| majority class in training:", y_train.mode()[0],
      f"| share of the majority class in test: {(y_test == y_train.mode()[0]).mean():.1%}")
test_results.round(3)

Test cases: 161 | majority class in training: 1 | share of the majority class in test: 59.0%


,Accuracy,Precision,Recall,F1,ROC-AUC,Cases flagged long
A. Majority baseline,0.590,0.590,1.000,0.742,0.500,161
B. Logistic Regression,0.646,0.694,0.716,0.705,0.659,98
C. Decision Tree,0.596,0.742,0.484,0.586,0.636,62


In [20]:
for name in ["B. Logistic Regression", "C. Decision Tree"]:
    print(name)
    print(pd.DataFrame(
        confusion_matrix(y_test, final_models[name].predict(X_test)),
        index=["actual 0 (not long)", "actual 1 (long)"],
        columns=["predicted 0", "predicted 1"],
    ))
    print()

B. Logistic Regression
                     predicted 0  predicted 1
actual 0 (not long)           36           30
actual 1 (long)               27           68

C. Decision Tree


                     predicted 0  predicted 1
actual 0 (not long)           50           16
actual 1 (long)               49           46



## 9. Interpretation

**A changed cohort, not only a later snapshot.** All long cases (span > 63 days) also last beyond day 30, so every long case
from the day-14 cohort is still here (377 training, 95 test). Only not-long cases whose span ended between day 14 and day 30
were removed. The share of long cases therefore rises to 55.7% (training) and 59.0% (test). That is why each model is
compared with **its own day-30 baseline**: the majority baseline already reaches 59.0% accuracy on the test cases, and
day-30 accuracy must not be compared directly with day-14 accuracy. The features also cover a longer window
(30 instead of 14 days), so day 14 versus day 30 is **not a perfectly controlled comparison**.

**Main measure: ROC-AUC** (how well the model ranks long cases above not-long cases; 0.50 = no better than chance).

| | Cross-validation (mean ± std) | Test |
|---|---|---|
| A. Majority baseline | 0.500 ± 0.000 | 0.500 |
| B. Logistic Regression | 0.702 ± 0.033 | 0.659 |
| C. Decision Tree | 0.629 ± 0.047 | 0.636 |

- **Logistic Regression** ranks cases better than its baseline in every one of the five folds (0.667–0.755) and on the
  test cases (0.659). The direction agrees between cross-validation and test, although the test value is a little lower.
- **The Decision Tree** is also above its baseline, but weaker and less stable than Logistic Regression.

**Warnings (precision and recall).** On the test cases, Logistic Regression flags 98 of 161 cases as long. 69.4% of those
flags are correct, compared with 59.0% if cases were flagged at random, and it finds 71.6% of the long cases (68 of 95). The
Decision Tree flags fewer cases (62): its flags are more often correct (74.2%) but it finds fewer than half of the long cases
(48.4%). The baseline flags everything, which gives the highest recall and F1 but no ability to tell cases apart.

**Answer.** Day-30 information gives a **modest** signal beyond its own baseline, mainly with Logistic Regression
(ROC-AUC about 0.66–0.70). The signal is clearer than at day 14, but it comes from a different cohort, the test set is
small (161 cases) and has been explored in earlier notebooks, so this is **exploratory evidence**, not proof that the model
would work on new cases, and it says nothing about usefulness in practice.

## 10. Limitations

- Exploratory study of **one Dutch municipality**; not SPAR or Montenegro data.
- The target is the **recorded activity span**, which may include publication, objection, archiving or later recording steps.
- Eligibility means the recorded span extends beyond day 30; it does not prove that the case was formally open.
- Event timestamps are assumed to show information available by day 30; the leakage check verifies the feature code, not this assumption.
- The original test set was already used in notebooks 07–09, and the 63-day threshold was chosen during exploratory
  analysis of all completed cases, including these test cases.
- The five-fold standard deviations describe variation between folds; they are not confidence intervals.